In [2]:
import os
import requests
from bs4 import BeautifulSoup
import re
import time

# 输入 ID 文件
input_file = r"C:\Users\localadmin\Desktop\其他重要\11.24\JGI\id.txt"

# 输出文件路径
folder_path = r"C:\Users\localadmin\Desktop\其他重要\11.24\JGI\Protein_sequences"
os.makedirs(folder_path, exist_ok=True)

output_file   = os.path.join(folder_path, "all.fasta")      # 保存序列
progress_file = os.path.join(folder_path, "progress.txt")   # 记录断点
failed_file   = os.path.join(folder_path, "failed.txt")     # 记录最终失败ID（多次重试后）

# 读取 gene id
with open(input_file, "r") as f:
    gene_ids = [line.strip() for line in f if line.strip()]

# 读取断点
start_index = 0
if os.path.exists(progress_file):
    with open(progress_file, "r") as pf:
        last_id = pf.read().strip()
        if last_id in gene_ids:
            start_index = gene_ids.index(last_id) + 1
            print(f"➡ 从断点继续: {last_id} 的下一个开始 (index={start_index})")
        else:
            print("⚠ progress.txt 中的 ID 未在 id.txt 中找到，将从头开始")
else:
    print("🆕 未发现断点，从头开始")

def fetch_fasta(gene_id, max_retries=3, sleep_sec=3):
    """
    返回：成功 -> 文本（FASTA）; 失败 -> None
    会自动重试 max_retries 次
    """
    url = f"https://img.jgi.doe.gov/cgi-bin/m/main.cgi?section=GeneDetail&page=genePageMainFaa&gene_oid={gene_id}"

    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, timeout=60)
            status = resp.status_code
        except Exception as e:
            print(f"  ⚠ 请求异常 {gene_id} (第 {attempt} 次): {e}")
            time.sleep(sleep_sec)
            continue

        if status != 200:
            print(f"  ⚠ 状态码异常 {gene_id}: {status} (第 {attempt} 次)")
            time.sleep(sleep_sec)
            continue

        html = resp.text
        soup = BeautifulSoup(html, "html.parser")
        pre = soup.find("pre")
        if not pre:
            print(f"  ⚠ 未找到 <pre> 标签 {gene_id} (可能是跳转/未登录/页面结构变化)")
            return None

        text = pre.get_text()
        text = re.sub(r"<.*?>", "", text).strip()

        if not text.startswith(">"):
            print(f"  ⚠ 内容不是FASTA格式（不以 '>' 开头）: {gene_id}")
            return None

        return text  # 成功

    # 多次重试仍失败
    return None

# 追加模式打开输出文件和失败文件
with open(output_file, "a", encoding="utf-8") as fasta_out, open(failed_file, "a", encoding="utf-8") as fail_out:
    for gid in gene_ids[start_index:]:
        print(f"\n=== 处理 {gid} ===")
        fasta = fetch_fasta(gid)

        if fasta:
            # 写入序列，立刻 flush，防止中途崩溃丢失
            fasta_out.write("\n" + fasta + "\n")
            fasta_out.flush()
            print(f"✔ 已保存 {gid}")

            # 更新断点
            with open(progress_file, "w", encoding="utf-8") as pf:
                pf.write(gid)
        else:
            print(f"✘ 失败 {gid} （多次尝试无结果）")
            fail_out.write(gid + "\n")
            fail_out.flush()

print("\n🎉 全部处理完成！结果已保存到：", output_file)
print("❗ 若还有 ID 在 failed.txt 中，可单独再写脚本对 failed.txt 里的 ID 重试。")


➡ 从断点继续: 650616877 的下一个开始 (index=1000)

🎉 全部处理完成！结果已保存到： C:\Users\localadmin\Desktop\其他重要\11.24\JGI\Protein_sequences\all.fasta
❗ 若还有 ID 在 failed.txt 中，可单独再写脚本对 failed.txt 里的 ID 重试。


In [3]:
import os
import requests
from bs4 import BeautifulSoup
import re
import time

# ========================= 配置区域 =========================

# 基础路径（和之前脚本保持一致）
base_folder = r"C:\Users\localadmin\Desktop\其他重要\11.24\JGI\Protein_sequences"

failed_file        = os.path.join(base_folder, "failed.txt")         # 第一次失败的ID
output_file        = os.path.join(base_folder, "all.fasta")         # 还是写到这个总fasta里
retry_failed_file  = os.path.join(base_folder, "failed_retry.txt")  # 重试后仍失败的ID
retry_progress_file = os.path.join(base_folder, "retry_progress.txt")  # 重试脚本自己的断点

# ==========================================================

# 读取失败的 gene id
if not os.path.exists(failed_file):
    raise FileNotFoundError(f"未找到 failed.txt：{failed_file}")

with open(failed_file, "r", encoding="utf-8") as f:
    gene_ids = [line.strip() for line in f if line.strip()]

print(f"共 {len(gene_ids)} 个失败ID需要重试")

# 读取重试断点
start_index = 0
if os.path.exists(retry_progress_file):
    with open(retry_progress_file, "r", encoding="utf-8") as pf:
        last_id = pf.read().strip()
        if last_id in gene_ids:
            start_index = gene_ids.index(last_id) + 1
            print(f"➡ 从重试断点继续: {last_id} 的下一个开始 (index={start_index})")
        else:
            print("⚠ retry_progress.txt 中的 ID 未在 failed.txt 中找到，将从头开始")
else:
    print("🆕 未发现重试断点，从头开始")

def fetch_fasta(gene_id, max_retries=3, sleep_sec=3):
    """
    返回：成功 -> 文本（FASTA）; 失败 -> None
    会自动重试 max_retries 次
    """
    url = f"https://img.jgi.doe.gov/cgi-bin/m/main.cgi?section=GeneDetail&page=genePageMainFaa&gene_oid={gene_id}"

    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, timeout=60)
            status = resp.status_code
        except Exception as e:
            print(f"  ⚠ 请求异常 {gene_id} (第 {attempt} 次): {e}")
            time.sleep(sleep_sec)
            continue

        if status != 200:
            print(f"  ⚠ 状态码异常 {gene_id}: {status} (第 {attempt} 次)")
            time.sleep(sleep_sec)
            continue

        html = resp.text
        soup = BeautifulSoup(html, "html.parser")
        pre = soup.find("pre")
        if not pre:
            print(f"  ⚠ 未找到 <pre> 标签 {gene_id} (可能是跳转/未登录/页面结构变化)")
            return None

        text = pre.get_text()
        text = re.sub(r"<.*?>", "", text).strip()

        if not text.startswith(">"):
            print(f"  ⚠ 内容不是FASTA格式（不以 '>' 开头）: {gene_id}")
            return None

        return text  # 成功

    # 多次重试仍失败
    return None


# 重试时，把“仍然失败”的写入新的 failed_retry.txt
with open(output_file, "a", encoding="utf-8") as fasta_out, \
     open(retry_failed_file, "a", encoding="utf-8") as fail_out:

    for gid in gene_ids[start_index:]:
        print(f"\n=== 重试 {gid} ===")
        fasta = fetch_fasta(gid)

        if fasta:
            # 写入序列，立刻 flush
            fasta_out.write("\n" + fasta + "\n")
            fasta_out.flush()
            print(f"✔ 重试成功，已保存 {gid}")

            # 更新重试断点
            with open(retry_progress_file, "w", encoding="utf-8") as pf:
                pf.write(gid)
        else:
            print(f"✘ 重试仍失败 {gid}")
            fail_out.write(gid + "\n")
            fail_out.flush()

print("\n🎉 失败ID重试完成！成功的已追加到：", output_file)
print("❗ 仍然失败的ID已写入：", retry_failed_file)


共 3 个失败ID需要重试
🆕 未发现重试断点，从头开始

=== 重试 638019684 ===
✔ 重试成功，已保存 638019684

=== 重试 638029271 ===
✔ 重试成功，已保存 638029271

=== 重试 644948812 ===
✔ 重试成功，已保存 644948812

🎉 失败ID重试完成！成功的已追加到： C:\Users\localadmin\Desktop\其他重要\11.24\JGI\Protein_sequences\all.fasta
❗ 仍然失败的ID已写入： C:\Users\localadmin\Desktop\其他重要\11.24\JGI\Protein_sequences\failed_retry.txt
